In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import  matplotlib.pyplot as plt
import ast
from dataclasses import dataclass
import random
from typing import NamedTuple

## 1.0 Dataset Generation

In [ ]:
DATA_DIR = Path("data")
OPL_PATH = DATA_DIR / "opl/"
GYM_600K_PATH = DATA_DIR / "600k_dataset/"


MIN_MEETS_FOR_AMPLITUDE = 5
PERIODIZATION_SHAPE = [
    0.00, 0.07, 0.14, 0.21,
    0.29, 0.37, 0.46, 0.55,
    0.65, 0.80, 1.00,
    0.20,
] # Normalized


LEVEL_SHAPE_PARAMS = {
    "novice":       {"floor": 0.65, "ceiling": 0.90},
    "intermediate": {"floor": 0.68, "ceiling": 0.93},
    "advanced":     {"floor": 0.72, "ceiling": 0.95},
    "elite":        {"floor": 0.75, "ceiling": 0.93},
}

LEVEL_MAP = {
    "elite":        {"advanced"},
    "advanced":     {"advanced"},
    "intermediate": {"intermediate"},
    "novice":       {"beginner", "novice"}
}

RPE_FLOOR = {
    "novice":       6.5,
    "intermediate": 7.0,
    "advanced":     7.5,
    "elite":        8.0,
}

RPE_CEILING = 9.5

BLOCK_WEEKS = 12
SESSIONS_PER_WEEK = 4

PHASE_BOUNDARIES = {
    "accumulation":    range(1, 5),
    "intensification": range(5, 9),
    "realisation":     range(9, 12),
    "deload":          range(12, 13),
}

_STRENGTH_GOAL_KEYWORDS = {
    "strength", "muscle", "hypertrophy", "powerlifting",
    "bulk", "mass", "gain", "power",
    "weightlifting",  
    "build",           
}

_BARBELL_EQUIPMENT_VALUES = {"Full Gym", "Garage Gym"}
ACCESSORIES_PER_SESSION = 4

OHP_BENCH_RATIO = {
    "novice":       0.61,
    "intermediate": 0.64,
    "advanced":     0.67,
    "elite":        0.70,
}

SESSION_FOCUS = {
    0: {"label": "Lower A",  "body_keywords": ["squat", "deadlift", "leg", "lunge", "hamstring", "glute", "hip", "calf", "quad"]},
    1: {"label": "Upper A",  "body_keywords": ["bench", "press", "chest", "shoulder", "tricep", "push", "dip", "fly"]},
    2: {"label": "Lower B",  "body_keywords": ["deadlift", "squat", "row", "pull", "back", "lat", "trap", "bicep", "curl"]},
    3: {"label": "Upper B",  "body_keywords": ["press", "shoulder", "ohp", "overhead", "push", "tricep", "chest", "dip"]},
}

### Helper Functions - helper_utils.py

In [471]:
def _parse_list_field(val) -> list[str]:
    """
    Convert list-literal strings to actual Python lists, normalised to lowercase.
    "['strength', 'hypertrophy']" → ["strength", "hypertrophy"]
    "strength"                    → ["strength"]
    "[]" / NaN / None             → []
    """
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    s = str(val).strip()
    if not s or s == "[]":
        return []
    if s.startswith("["):
        try:
            result = ast.literal_eval(s)
            return [str(v).strip().lower() for v in result if str(v).strip()]
        except (ValueError, SyntaxError):
            return []
    return [s.lower()]



def _classify_DOTS(dots: float) -> str:

    DOTS_THRESHOLDS = {
        "elite":        510,
        "advanced":     410,
        "intermediate": 310,
    }

    if dots >= DOTS_THRESHOLDS['elite']:
        return "elite"
    if dots >= DOTS_THRESHOLDS['advanced']:
        return 'advanced'
    if dots >= DOTS_THRESHOLDS['intermediate']:
        return 'intermediate'

    return "novice"

def _derive_opl_amplitude(opl_df: pd.DataFrame, MIN_MEETS_FOR_AMPLITUDE: int) -> dict[str, float]:

    meet_counts = opl_df.groupby("Name")['Date'].count()
    qualified = meet_counts[meet_counts >= MIN_MEETS_FOR_AMPLITUDE].index
    df = opl_df[opl_df['Name'].isin(qualified)].copy()
    print(f"[INFO-OPL-Shape] - Minimum {MIN_MEETS_FOR_AMPLITUDE} Meets Filtered OPL Dataset Shape: {df.shape}")

    peak_dots = df.groupby("Name")['Dots'].max().rename('peak_dots')
    df = df.join(peak_dots, on = 'Name')
    df['training_level'] = df['peak_dots'].apply(_classify_DOTS)

    df['prev_total'] = df.sort_values(['Name', 'Date'])['TotalKg'].shift(1)
    df = df.dropna(subset = ['prev_total', 'TotalKg'])
    df = df[df['prev_total'] > 0]
    df['progression'] = (df['TotalKg'] - df['prev_total']) / df['prev_total']
    df = df[df['progression'] > 0]
    df = df[df["progression"] <= 0.20]

    print(f"[INFO-OPL-Shape] - Increasing Progression Filtered OPL Dataset Shape: {df.shape}")


    amplitude = (
        df.groupby('training_level')['progression']
        .mean()
        .to_dict()
    )

    counts = df.groupby("training_level")["progression"].count().to_dict()
    for level, val in amplitude.items():
        n = counts.get(level, 0)
        print(f"  [OPL amplitude] {level}: {val} ({n} progressions)")
    
    return amplitude



def _derive_strength(goals: pd.Series) -> set:

    all_goals = {g for goal in goals for g in goal}
    strength = {g for g in all_goals if any(kw in g for kw in _STRENGTH_GOAL_KEYWORDS)}
    print(f"  [goals]     {len(all_goals):,} unique goal strings found in dataset")
    print(f"  [goals]     {len(strength):,} classified as strength-relevant: {sorted(strength)}")
    return strength



def _main_lift_for_day(day_index: int, persona: AthletePersona) -> tuple[str, float]:
    ohp_peak = persona.bench_peak_kg * OHP_BENCH_RATIO[persona.training_level]
    return {
        0: ("Squat",    persona.squat_peak_kg),
        1: ("Bench",    persona.bench_peak_kg),
        2: ("Deadlift", persona.deadlift_peak_kg),
        3: ("OHP",      ohp_peak),
    }[day_index % 4]
 

def _week_to_phase(week: int) -> str:
    for phase, weeks in PHASE_BOUNDARIES.items():
        if week in weeks:
            return phase
    return "deload"

In [500]:
@dataclass
class PeriodizationTemplate:
    training_level: str
    week_pcts:      list[float]
    rpe_curve:      list[float]
    amplitude:      float
    amp_source:     int           # number of OPL athletes contributing

@dataclass
class AthletePersona:
    athlete_id: str
    sex: str
    age: float
    bodyweight_kg: float
    weight_class_kg: float
    squat_peak_kg: float
    bench_peak_kg: float
    deadlift_peak_kg: float
    total_kg: float
    dots: float
    training_level: str


@dataclass
class Exercise:
    name: str
    goal: str
    equipment: str
    intensity: float    
    sets: int
    reps_value: float   
    reps_unit: str      # "reps" or "seconds"
    level: str

@dataclass
class SessionLog:
    """One training session within a 12-week block."""
    week:           int
    day_index:      int
    day_label:      str
    main_lift:      str
    main_lift_kg:   float
    main_lift_rpe:  float
    volume_pct:     float
    block_phase:    str    # accumulation / intensification / realisation / deload
    accessories:    list[Exercise]
 
@dataclass
class AthleteRecord:
    """Complete generated record for one athlete: persona + full 12-week block."""
    persona:  AthletePersona
    sessions: list[SessionLog]
 
 
 

### Dataset Creation - data_generation.py

#### 1.1 OPL Dataset

In [292]:
def get_opl_dataset() -> pd.DataFrame:

    # Only Grab needed columns
    check_OPL_exist = OPL_PATH / 'updated_opl.csv'

    if check_OPL_exist.is_file():

        print("INFO-OPL - Found Cleaned OPL Dataset")
        df = pd.read_csv(check_OPL_exist)
        print(f"[INFO-OPL-Shape] Cleaned OPL Dataset Shape: {df.shape}")
        return df


    else:

        print("[INFO-OPL] - Reading original OPL, Cleaning dataset and Saving")
        cols = ['Name', 'Sex', 'Age', 'BodyweightKg', 'WeightClassKg', 'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'TotalKg', 'Dots', 'Date'] 
        df = pd.read_csv(OPL_PATH / 'openpowerlifting.csv', usecols = lambda c : c in cols, low_memory = False)
        print(f"[INFO-OPL-Shape] Original OPL Dataset Shape: {df.shape}")

        # Keep Valid Data only
        df = df.dropna(subset = ['Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'Dots'])
        for lift in ['Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg', 'Dots']:
            df = df[df[lift] > 0]

        df = df[df["Dots"] > 0]
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.reset_index(drop = True)

        print(f"[INFO-OPL-Shape] Cleaned OPL Dataset Shape: {df.shape}")
        df.to_csv(OPL_PATH / 'updated_opl.csv', index = False)
        print(f"[INFO-OPL] Cleaned OPL Dataset Saved to: {OPL_PATH / 'updated_opl.csv'}")

        return df


opl_df = get_opl_dataset()
amplitude = _derive_opl_amplitude(opl_df, MIN_MEETS_FOR_AMPLITUDE)

INFO-OPL - Found Cleaned OPL Dataset
[INFO-OPL-Shape] Cleaned OPL Dataset Shape: (1928369, 11)
[INFO-OPL-Shape] - Minimum 5 Meets Filtered OPL Dataset Shape: (1097509, 11)
[INFO-OPL-Shape] - Increasing Progression Filtered OPL Dataset Shape: (598117, 15)
  [OPL amplitude] advanced: 0.05531041988474781 (222903 progressions)
  [OPL amplitude] elite: 0.05388790890474626 (68840 progressions)
  [OPL amplitude] intermediate: 0.05696038921262884 (249831 progressions)
  [OPL amplitude] novice: 0.06335108989772474 (56543 progressions)


#### 1.2 PeriodizationTemplate

In [344]:
def build_periodization_templates(
        opl_df: pd.DataFrame,
        n_weeks: int = BLOCK_WEEKS
) -> dict[str, PeriodizationTemplate]:
    

    opl_copy = opl_df.copy()
    opl_copy['training_level'] = opl_copy['Dots'].apply(_classify_DOTS)

    athletes_per_level = (
        opl_copy.groupby("training_level")['Name']
        .nunique().
        to_dict()
    )

    templates: dict[str, PeriodizationTemplate] = {}

    for level in ['novice', 'intermediate', 'advanced', 'elite']:
        amplitude_level = amplitude[level]
        params = LEVEL_SHAPE_PARAMS[level]
        floor = params['floor']
        ceiling = params['ceiling']


        week_pcts = [round(float(floor + s * (ceiling - floor)), 4)
                    for s in PERIODIZATION_SHAPE]

        rpe_curve = [
            round(float(RPE_FLOOR[level] + s * (RPE_CEILING - RPE_FLOOR[level])), 3)
            for s in PERIODIZATION_SHAPE
        ]
        templates[level] = PeriodizationTemplate(
            training_level = level,
            week_pcts = week_pcts,
            rpe_curve = rpe_curve,
            amplitude = amplitude_level,
            amp_source = athletes_per_level[level]
        )
    return templates
        
templates = build_periodization_templates(opl_df)
templates

{'novice': PeriodizationTemplate(training_level='novice', week_pcts=[0.65, 0.6675, 0.685, 0.7025, 0.7225, 0.7425, 0.765, 0.7875, 0.8125, 0.85, 0.9, 0.7], rpe_curve=[6.5, 6.71, 6.92, 7.13, 7.37, 7.61, 7.88, 8.15, 8.45, 8.9, 9.5, 7.1], amplitude=0.06335108989772474, amp_source=297593),
 'intermediate': PeriodizationTemplate(training_level='intermediate', week_pcts=[0.68, 0.6975, 0.715, 0.7325, 0.7525, 0.7725, 0.795, 0.8175, 0.8425, 0.88, 0.93, 0.73], rpe_curve=[7.0, 7.175, 7.35, 7.525, 7.725, 7.925, 8.15, 8.375, 8.625, 9.0, 9.5, 7.5], amplitude=0.05696038921262884, amp_source=335812),
 'advanced': PeriodizationTemplate(training_level='advanced', week_pcts=[0.72, 0.7361, 0.7522, 0.7683, 0.7867, 0.8051, 0.8258, 0.8465, 0.8695, 0.904, 0.95, 0.766], rpe_curve=[7.5, 7.64, 7.78, 7.92, 8.08, 8.24, 8.42, 8.6, 8.8, 9.1, 9.5, 7.9], amplitude=0.05531041988474781, amp_source=106581),
 'elite': PeriodizationTemplate(training_level='elite', week_pcts=[0.75, 0.7626, 0.7752, 0.7878, 0.8022, 0.8166, 0.83

#### 1.3 600k Dataset

In [458]:
class GymData(NamedTuple):

    df:             pd.DataFrame
    strength_goals: set
    barbell_equip:  set


def get_600k_dataset() -> pd.DataFrame:

    check_600k_exist = GYM_600K_PATH / 'combined_600k_dataset.csv'

    if check_600k_exist.is_file():

        print("[INFO-600K-Combined] - Found Combined 600k Dataset")
        ex_df = pd.read_csv(check_600k_exist)
        ex_df["goal"]  = ex_df["goal"].apply(_parse_list_field)
        ex_df["level"] = ex_df["level"].apply(_parse_list_field)
        print(f"[INFO-600K-Combined] Combined Cleaned 600k Dataset Shape: {ex_df.shape}")
        # return ex_df

    else:
        print("[INFO-600K-Combined] - Reading Both datasets, Cleaning datasets and Saving")

        check_600k_EX_exist = GYM_600K_PATH / 'programs_detailed_boostcamp_kaggle.csv'
        check_600k_PROG_exist = GYM_600K_PATH / 'program_summary.csv'

        # ----------------------------------------------------------------------------------------------------------
        # Read 600K Original Datasets 

        ex_df = pd.read_csv(check_600k_EX_exist, low_memory = False)
        prog_df = pd.read_csv(check_600k_PROG_exist, low_memory = False)

        print("[INFO-600k-EX] - Reading original 600K Fitness Exercises data")
        print(f"[INFO-600k-EX] Original 600K Fitness Exercises Dataset Shape: {ex_df.shape}")

        print("[INFO-600k-PROG] - Reading original 600K Program Summary data")
        print(f"[INFO-600k-PROG] Original 600K Program Summary Dataset Shape: {prog_df.shape}")

        # ----------------------------------------------------------------------------------------------------------
        # Drop Goal less Programs and corresponding exercises 

        prog_df = prog_df[prog_df['goal'] != '[]'].reset_index(drop = True)
        prog_df = prog_df[prog_df['equipment'] != '[]'].reset_index(drop = True)
        prog_df = prog_df[prog_df['equipment'].notna()].reset_index(drop = True)
        prog_df = prog_df[prog_df['description'].notna()].reset_index(drop = True)
        prog_df = prog_df[prog_df['last_edit'].notna()].reset_index(drop = True)


        valid_titles = set(prog_df['title'])
        ex_df = ex_df[ex_df['title'].isin(valid_titles)].reset_index(drop = True)

        print(f"[INFO-600k_EX] - Dropped Goalless Exercises - Shape: {ex_df.shape}")
        print(f"[INFO-600k_PROG] - Dropped Goalless program - Shape: {prog_df.shape}")

        # ----------------------------------------------------------------------------------------------------------
        # Join Program and Exercises dataset onto exercises rows

        prog_slim = (
            prog_df[["title", "goal", "equipment"]]
            .drop_duplicates("title")
            .rename(columns={"goal": "goal_prog", "equipment": "equipment_prog"})
        )

        ex_df = ex_df.merge(prog_slim, on="title", how="left")

        # ----------------------------------------------------------------------------------------------------------
        # Fill Program level goals and equipment onto exercises goals 

        for col, prog_col in [("goal", "goal_prog"), ("equipment", "equipment_prog")]:
            if prog_col in ex_df.columns:
                ex_df[col] = ex_df[prog_col].fillna(ex_df[col])
                ex_df.drop(columns=[prog_col], inplace=True)

        # ----------------------------------------------------------------------------------------------------------
        # Convert list literal string to Python lists

        ex_df['goal'] = ex_df['goal'].apply(_parse_list_field)
        ex_df['level'] = ex_df['level'].apply(_parse_list_field)
        ex_df = ex_df[ex_df["equipment"].isin(_BARBELL_EQUIPMENT_VALUES)].reset_index(drop=True)


        # ----------------------------------------------------------------------------------------------------------
        # Convert Sets and Reps to int

        # ex_df["sets"] = pd.to_numeric(ex_df["sets"], errors="coerce").astype(int)
        ex_df["reps"] = pd.to_numeric(ex_df["reps"], errors="coerce")
        ex_df['intensity'] = pd.to_numeric(ex_df['intensity'], errors = "coerce").fillna(0.0)
        # ex_df = ex_df[ex_df['level'] != '[]'].reset_index(drop = True)

        print(f"[INFO-600K-Combined] Final 600k dataset Shape]: {ex_df.shape}")
        ex_df.to_csv(GYM_600K_PATH / 'combined_600k_dataset.csv', index = False)

    strength_goals = _derive_strength(ex_df["goal"])
    barbell_equip  = _BARBELL_EQUIPMENT_VALUES

    return GymData(df = ex_df, strength_goals = strength_goals, barbell_equip = barbell_equip)

gym_df = get_600k_dataset()

[INFO-600K-Combined] - Reading Both datasets, Cleaning datasets and Saving
[INFO-600k-EX] - Reading original 600K Fitness Exercises data
[INFO-600k-EX] Original 600K Fitness Exercises Dataset Shape: (605033, 16)
[INFO-600k-PROG] - Reading original 600K Program Summary data
[INFO-600k-PROG] Original 600K Program Summary Dataset Shape: (2598, 10)
[INFO-600k_EX] - Dropped Goalless Exercises - Shape: (601719, 16)
[INFO-600k_PROG] - Dropped Goalless program - Shape: (2584, 10)
[INFO-600K-Combined] Final 600k dataset Shape]: (559089, 16)
  [goals]     7 unique goal strings found in dataset
  [goals]     5 classified as strength-relevant: ['bodybuilding', 'muscle & sculpting', 'olympic weightlifting', 'powerbuilding', 'powerlifting']


#### 1.4 Persona Sampling

In [488]:
def sample_athlete_persona(opl_df: pd.DataFrame, athlete_id: str) -> AthletePersona:
    
    persona_df = opl_df.drop(
        columns = [c for c in ['Name', 'Date'] if c in opl_df.columns]
    )
    seed = int.from_bytes(athlete_id.encode(), "big") % (2 ** 31)
    row = persona_df.iloc[random.Random(seed).randint(0, len(persona_df) - 1)]
    dots = float(row['Dots'])

    return AthletePersona(
        athlete_id = athlete_id,
        sex = str(row['Sex']),
        age = float(row['Age']),
        bodyweight_kg = float(row['BodyweightKg']),
        weight_class_kg = float(row['WeightClassKg']),
        squat_peak_kg = float(row['Best3SquatKg']),
        bench_peak_kg = float(row['Best3BenchKg']),
        deadlift_peak_kg = float(row['Best3DeadliftKg']),
        total_kg = float(row['TotalKg']),
        dots = float(row['Dots']),
        training_level = _classify_DOTS(dots)
    )

#### 1.5 Accessory Selection

In [489]:
def query_accessories(
        gym_df: pd.DataFrame,
        training_level: str,
        strength_goals: set,
        barbell_equip: set,
        day_index: int, 
        n: int = ACCESSORIES_PER_SESSION,
        seed: int = 0,

) -> list[Exercise]:
    
    level_targets = LEVEL_MAP[training_level]
    level_label = sorted(level_targets)[0]
    level_mask = gym_df['level'].apply(lambda x: bool(set(x) & level_targets))
    goal_mask = gym_df['goal'].apply(lambda x: bool(set(x) & strength_goals))
    equip_mask = gym_df['equipment'].str.lower().str.strip().isin(barbell_equip)

    body_keywords = SESSION_FOCUS[day_index]['body_keywords']
    body_mask = gym_df['exercise_name'].str.lower().apply(
        lambda name: any(kw in name for kw in body_keywords)
    )

    filtered = pd.DataFrame()
    for mask in [
        body_mask & level_mask & goal_mask & equip_mask,  # step 0 — tightest
        body_mask & level_mask & goal_mask,                # step 1
        body_mask & goal_mask,                             # step 2
        body_mask,                                         # step 3
        level_mask & goal_mask & equip_mask,               # step 4 — fallback without body
        level_mask & goal_mask,                            # step 5
        goal_mask,                                         # step 6
        pd.Series(True, index=gym_df.index),               # step 7 — last resort
    ]:
        filtered = gym_df[mask]
        if not filtered.empty:
            break

    
    filtered = filtered.copy()
    filtered["_intensity_sort"] = filtered["intensity"].where(
        filtered["intensity"] > 0, -1
    )
    filtered = filtered.sort_values("_intensity_sort", ascending=False)
 
    filtered = filtered.dropna(subset=["exercise_name", "reps"])
    sampled  = filtered.head(n * 3).sample(frac=1, random_state=seed).head(n)
 
    exercises = []
    for _, row in sampled.iterrows():
        raw_reps = float(row["reps"])
        exercises.append(Exercise(
            name       = str(row["exercise_name"]),
            goal       = ", ".join(row["goal"]) if row["goal"] else "",
            equipment  = str(row.get("equipment", "")),
            intensity  = float(row.get("intensity", 0.0)),
            sets       = int(row.get("sets", 3)),
            reps_value = abs(raw_reps),
            reps_unit  = "seconds" if raw_reps < 0 else "reps",
            level      = level_label,
        ))
    return exercises


In [490]:
aid = "athlete_000001"
persona = sample_athlete_persona(opl_df, aid)
persona

AthletePersona(athlete_id='athlete_000001', sex='M', age=18.0, bodyweight_kg=70.5, weight_class_kg=75.0, squat_peak_kg=190.0, bench_peak_kg=117.5, deadlift_peak_kg=220.0, total_kg=527.5, dots=394.29, training_level='intermediate')

In [492]:
temp = query_accessories(gym_df.df, 
                persona.training_level,
                strength_goals = gym_df.strength_goals,
                barbell_equip = gym_df.barbell_equip,
                day_index = 0,
                seed = 0
                )

In [493]:
temp

[Exercise(name='Deadlift (Paused)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=2, reps_value=12.0, reps_unit='reps', level='intermediate'),
 Exercise(name='Walking Lunge (Dumbbell)', goal='bodybuilding, muscle & sculpting', equipment='Full Gym', intensity=10.0, sets=2, reps_value=4.0, reps_unit='reps', level='intermediate'),
 Exercise(name='Tempo Squat (Barbell)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=3, reps_value=12.0, reps_unit='reps', level='intermediate'),
 Exercise(name='Deadlift (Paused)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=2, reps_value=12.0, reps_unit='reps', level='intermediate')]

In [508]:
def build_training_block(
        persona: AthletePersona,
        gym_df: pd.DataFrame,
        template: PeriodizationTemplate,
        strength_goals: set,
        barbell_equip: set,
        n_weeks: int = BLOCK_WEEKS,
        sessions_per_week: int = SESSIONS_PER_WEEK,
        seed: int = 0
) -> list[SessionLog]:

    sessions = []
    for week in range(1, n_weeks + 1):
        vol_pct = template.week_pcts[week - 1]
        rpe = template.rpe_curve[week - 1]
        block_phase = _week_to_phase(week)


        for day_index in range(sessions_per_week):
            lift_name, competition_1rm = _main_lift_for_day(day_index, persona)
            block_target_kg = competition_1rm * (1.0 + template.amplitude)
            working_kg = round(block_target_kg * vol_pct / 2.5) * 2.5

            sessions.append(SessionLog(
                week = week,
                day_index = day_index,
                day_label = SESSION_FOCUS[day_index]['label'],
                main_lift = lift_name,
                main_lift_kg = working_kg,
                main_lift_rpe = rpe,
                volume_pct = vol_pct,
                block_phase = block_phase,
                accessories = query_accessories(
                    gym_df, 
                    training_level = persona.training_level,
                    strength_goals = strength_goals,
                    barbell_equip = barbell_equip,
                    day_index = day_index,
                    n = ACCESSORIES_PER_SESSION,
                    seed = seed + week * 100 + day_index
                )
            ))

    return sessions

In [509]:
TB = build_training_block(persona, 
                     gym_df.df, 
                     templates[persona.training_level], 
                     gym_df.strength_goals, 
                     gym_df.barbell_equip,
                     )

In [512]:
def generate_one_athlete(
        athlete_id: str,
        opl_df: pd.DataFrame,
        gym_df: pd.DataFrame,
        templates: dict[str, PeriodizationTemplate],
        strength_goals: set,
        barbell_equip: set
) -> AthleteRecord:
    
    BIG_SEED = (2 ** 31)
    persona = sample_athlete_persona(opl_df, athlete_id)
    template = templates[persona.training_level]
    sessions = build_training_block(
        persona,
        gym_df,
        template,
        strength_goals = strength_goals,
        barbell_equip = barbell_equip,
        seed = int.from_bytes(athlete_id.encode(), "big") % BIG_SEED
    )

    return AthleteRecord(persona = persona,
                         sessions = sessions)

In [513]:
ath_record = generate_one_athlete(athlete_id = aid,
                     opl_df = opl_df,
                     gym_df = gym_df.df,
                     templates = templates,
                     strength_goals = gym_df.strength_goals,
                     barbell_equip = gym_df.barbell_equip)

ath_record

AthleteRecord(persona=AthletePersona(athlete_id='athlete_000001', sex='M', age=18.0, bodyweight_kg=70.5, weight_class_kg=75.0, squat_peak_kg=190.0, bench_peak_kg=117.5, deadlift_peak_kg=220.0, total_kg=527.5, dots=394.29, training_level='intermediate'), sessions=[SessionLog(week=1, day_index=0, day_label='Lower A', main_lift='Squat', main_lift_kg=137.5, main_lift_rpe=7.0, volume_pct=0.68, block_phase='accumulation', accessories=[Exercise(name='Deadlift (Paused)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=2, reps_value=12.0, reps_unit='reps', level='intermediate'), Exercise(name='Deadlift (Barbell)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=3, reps_value=12.0, reps_unit='reps', level='intermediate'), Exercise(name='Deadlift (Barbell)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment

In [514]:
ath_record.sessions

[SessionLog(week=1, day_index=0, day_label='Lower A', main_lift='Squat', main_lift_kg=137.5, main_lift_rpe=7.0, volume_pct=0.68, block_phase='accumulation', accessories=[Exercise(name='Deadlift (Paused)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=2, reps_value=12.0, reps_unit='reps', level='intermediate'), Exercise(name='Deadlift (Barbell)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=3, reps_value=12.0, reps_unit='reps', level='intermediate'), Exercise(name='Deadlift (Barbell)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=3, reps_value=10.0, reps_unit='reps', level='intermediate'), Exercise(name='Squat (Paused)', goal='athletics, bodybuilding, powerbuilding, muscle & sculpting, powerlifting', equipment='Full Gym', intensity=10.0, sets=3, reps_valu

## 0.0 TEST

In [1]:
from config.settings import(
    N_ATHLETES,
    CHECKPOINT_THRESHOLD,
    OPL_PATH,
    GYM_EX_PATH,
    GYM_PROG_PATH,
    SESSIONS_PATH,
    BLOCK_SUMMARY_PATH,
    CHECKPOINT_PATH, 
    OUT_DIR
)
from pipeline.dataset.opl_loader import get_opl_dataset, _classify_DOTS


In [8]:
def chunk_text(
        text : str,
        chunk_size : int = 1000,
        chunk_overlap : int = 100
) -> list[str]:
    
    text = text.strip()
    if not text: return []


    chunks : list[str] = []
    start = 0

    while start < len(text):
        end  = min(start + chunk_size, len(text))
        chunk = text[start:end]

        if end < len(text):
            last_space = chunk.rfind(" ")
            if last_space > chunk_overlap:
                end = start + last_space
                chunk = text[start:end]

        chunks.append(chunk)
        start = end - chunk_overlap

        if end >= len(text):
            break

    return [c for c in chunks if c]


_PHASE_LABEL = {
    "accumulation":    "Accumulation",
    "intensification": "Intensification",
    "realisation":     "Realisation",
    "deload":          "Deload",
}

In [124]:
df = pd.read_csv(SESSIONS_PATH)
df.columns

Index(['athlete_id', 'opl_row_index', 'primary_program', 'secondary_program',
       'sex', 'age', 'bodyweight_kg', 'weight_class_kg', 'squat_peak_kg',
       'bench_peak_kg', 'deadlift_peak_kg', 'total_kg', 'dots',
       'training_level', 'week', 'day_index', 'day_label', 'block_phase',
       'main_lift', 'main_lift_kg', 'main_lift_pct_of_peak', 'main_lift_rpe',
       'volume_pct', 'accessories', 'accessory_titles', 'accessory_sets',
       'accessory_reps', 'accessory_reps_unit', 'accessory_intensity',
       'main_lift_delta_kg'],
      dtype='str')

In [125]:
def session_to_nl(
        df: pd.DataFrame,
        athlete_id : str,
        week : int,
):
    wk_rows = df[
        (df['athlete_id'] == athlete_id) &
        (df['week'] == week)
        ]
    
    if wk_rows.empty: return ""

    phase_raw = str(wk_rows['block_phase'].iloc[0])
    phase_label = _PHASE_LABEL.get(phase_raw, phase_raw.capitalize())

    lines : list[str] = [
        f"Athlete {athlete_id}  |   Week {week}  |  {phase_label}"
    ]

    lift_order = ['Squat', 'Bench', 'Deadlift', 'OHP']
    for lift in lift_order:
        # Get lift row
        row = wk_rows[wk_rows['main_lift'] == lift]
        # Check for empty row
        if row.empty: continue
        # get first row, extract kg, rpe, delta, pct
        r = row.iloc[0]
        kg = r['main_lift_kg']
        rpe = r['main_lift_rpe']
        delta = r.get('main_lift_delta_kg', None)
        pct = r.get('main_lift_pct_of_peak', None)

        delta_str = ""

        # Check if delta exists and not 0
        if pd.notna(delta) and delta != 0:
            sign = "+" if delta > 0 else ""
            delta_str = f" ({sign}{delta:.1f}kg vs prev week)"

        pct_str = f"  [{pct*100:.1f}% of 1RM]" if pd.notna(pct) else ""
        lines.append(
            f"{lift}: {kg:.1f}kg{delta_str}  |  RPE {rpe:.1f}{pct_str}"
        )
    # print(lines)
    # print(wk_rows['day_label'])
    day_order = ['Lower A', 'Upper A', 'Lower B', 'Upper B']
    for day_label in day_order:
        # get label row
        row = wk_rows[wk_rows['day_label'] == day_label]
        if row.empty: continue
        r = row.iloc[0]


        names = [x.strip() for x in str(r.get("accessories",            "")).split("|")]
        sets_ = [x.strip() for x in str(r.get("accessory_sets",         "")).split("|")]
        reps_ = [x.strip() for x in str(r.get("accessory_reps",         "")).split("|")]
        units_ = [x.strip() for x in str(r.get("accessory_reps_unit",   "")).split("|")]


        acc_parts : list[str] = []
        for i, name in enumerate(names):
            s = sets_[i]    if i < len(sets_) else "?"
            rep = reps_[i]  if i < len(reps_) else "?"
            u = units_[i]  if i < len(units_) else "?"

            try: 
                rv = float(rep)
                r_fmt = f"{int(rv)}" if rv == int(rv) else f"{rv:.1f}"
            except:
                r_fmt = rep
            sfx = "s" if u == 'seconds' else ""
            acc_parts.append(f"{name} {s} x {r_fmt}{sfx}")


        if acc_parts:
            lines.append(f"Accessories ({day_label}): {' | '.join(acc_parts)}")
    # print(lines)
    return "\n".join(lines)

In [138]:
def build_all_nl_strings(
        session_df: pd.DataFrame,
) -> list[dict]:
    
    records: list[dict] = []

    for athlete_id in session_df['athlete_id'].unique():
        athlete = session_df[session_df['athlete_id'] == athlete_id]

        meta = {
            'athlete_id': athlete_id,
            'training_level': session_df['training_level'].iloc[0],
            'dots': float(session_df['dots'].iloc[0]),
            'opl_row_index': int(athlete['opl_row_index'].iloc[0]) 
                        if 'opl_row_index' in athlete.columns else -1,
            'primary_program': str(athlete['primary_program'].iloc[0])
                        if 'primary_program' in athlete.columns else ""
        }

        for week in sorted(athlete['week'].unique()):
            phase = str(athlete[athlete['week'] == week]['block_phase'].iloc[0])
            text = session_to_nl(session_df, athlete_id, int(week))
            if not text: continue
            records.append({
                **meta,
                'week': int(week),
                'block_phase': phase,
                'text': text
            })

        return records

In [139]:
temp = df[0:16]
records = build_all_nl_strings(temp)

In [140]:
records

[{'athlete_id': 'athlete_00000',
  'training_level': 'novice',
  'dots': 294.95,
  'opl_row_index': 519910,
  'primary_program': 'Powerlifting Hypertrophy Block',
  'week': 1,
  'block_phase': 'accumulation',
  'text': 'Athlete athlete_00000  |   Week 1  |  Accumulation\nSquat: 112.5kg  |  RPE 6.5  [69.2% of 1RM]\nBench: 67.5kg  |  RPE 6.5  [69.2% of 1RM]\nDeadlift: 132.5kg  |  RPE 6.5  [69.7% of 1RM]\nOHP: 40.0kg  |  RPE 6.5  [67.3% of 1RM]\nAccessories (Lower A): Glute-Biased Leg Press 3 x 15 | Sissy Squats (Partial ROM) 3 x 8 | ATG Split Squat 3 x 12 | Smith Machine Reverse Lunges 1 x 10\nAccessories (Upper A): Chest Supported Row (Smith Machine) 1 x 12 | Bench Press Incline (Close Grip) 1 x 12 | Incline Dumbbell Row 2 x 5 | Overhead Single Arm Rope Tricep Extension 2 x 8\nAccessories (Lower B): Halting Deadlift 2 x 12 | Deadlift (Smith Machine) 1 x 12 | Hip Thrust (Band) 3 x 12 | Tempo Deadlift 4 x 5\nAccessories (Upper B): Lat Pulldown (Close Grip) 3 x 15 | Shoulder Press Smith ma